# UE Data Mining - Projet

Notebook à compléter dans le cadre du projet de l’UE Data Mining des Master 2 Intelligence Artificielle et Master 2 Data Science de l’année scolaire 2026-2027.

* Nom du groupe : 12
* Membres du groupe :
    * idil MAHAMOUD GUEDI
    * André da Conceição
    * Ali Chabaka 
    * Aissam Eddine Boukhelkhal
    * Lucas Duchamp
    * RATOANDROMANANA Iavosoa Miella


## Pensez à mettre des titres

Et du texte pour expliquer les différentes étapes de votre raisonnement, les résultats obtenus et les conclusions que vous en avez tiré.

Pour rappel, ce notebook sera aussi votre rapport sur ce projet, donc soyez précis et rigoureux dans ce que vous décrivez et écrivez.

In [1]:
### Data Cleaning
### Ecology
import pandas as pd
df_bats = pd.read_csv('data/ecology/bat_surveys.csv')

df_bats.describe()

,colony_estimate,comparison_species_count
count,4994.000000,4994.000000
mean,15.826512,10.711854
std,9.603602,3.449512
min,0.000000,1.000000
25%,9.300000,8.000000
50%,14.200000,11.000000
75%,20.500000,13.000000
max,99.600000,26.000000


In [2]:
# On enlève les lignes doublons (il y en a 2)
df_bats_clean = df_bats.duplicated()
print(df_bats_clean.sum())
df_bats_clean = df_bats.drop_duplicates()

2


In [3]:
# On remplace les noms des observer team comme "inconnue")
print(df_bats_clean.isna().sum())
df_bats_clean['observer_team'] = df_bats_clean['observer_team'].fillna('inconnue')

In [4]:
for c in df_bats_clean:
    print(df_bats_clean[c].value_counts())

# On observe problèmes de generalisation de dates, de ponctuations (floats), de majuscules etc
# On a aussi la survey_quality à mettre en 2 ou 3 colonnes

survey_id
BAT-0000001    1
BAT-0000002    1
BAT-0000003    1
BAT-0000004    1
BAT-0000005    1
              ..
BAT-0004988    1
BAT-0004989    1
BAT-0004990    1
BAT-0004991    1
BAT-0004992    1
Name: count, Length: 4992, dtype: int64
site_code
KNT-KES    78
KNT-LUM    78
KNT-SEN    78
AVL-MAR    78
AVL-VEL    78
           ..
TMB-WST    78
TMB-SXP    78
TMB-WQL    78
TMB-QCB    78
TMB-DLQ    78
Name: count, Length: 64, dtype: int64
survey_start
2019-01-05T18:00:00    64
2019-02-05T18:00:00    64
2019-03-05T18:00:00    64
2019-04-05T18:00:00    64
2019-05-05T18:00:00    64
                       ..
2025-02-05T18:00:00    64
2025-03-05T18:00:00    64
2025-04-05T18:00:00    64
2025-05-05T18:00:00    64
2025-06-05T18:00:00    64
Name: count, Length: 78, dtype: int64
survey_end
2019-01-12T00:00:00     64
2019-03-12T00:00:00     64
2019-05-12T00:00:00     64
2019-06-12T00:00:00     64
2019-07-12T00:00:00     64
                        ..
12/07/2021 00:00:00      1
12-Apr-2019 00:00:00    

In [5]:

def clean_numeric(val):
    if pd.isna(val):
        return None
    s = str(val).strip()
    # on enleve les lettres
    s = s.replace(',', '.')
    # on remplace les virgules
    try:
        return float(s)
    except ValueError:
        return None
for c in ["observation_hours","observed_bat_count","acoustic_detection_count","nightly_visit_count","colony_estimate","comparison_species_count","weather_interference"]:
    df_bats_clean[c] = df_bats_clean[c].apply(clean_numeric)
print(df_bats_clean)

        survey_id site_code         survey_start           survey_end  \
0     BAT-0000001   KNT-KES  2019-01-05T18:00:00  2019-01-12T00:00:00   
1     BAT-0000002   KNT-KES  2019-02-05T18:00:00  2019-02-12T00:00:00   
2     BAT-0000003   KNT-KES  2019-03-05T18:00:00  2019-03-12T00:00:00   
3     BAT-0000004   KNT-KES  2019-04-05T18:00:00  2019-04-12T00:00:00   
4     BAT-0000005   KNT-KES  2019-05-05T18:00:00  2019-05-12T00:00:00   
...           ...       ...                  ...                  ...   
4987  BAT-0004988   TMB-DLQ  2025-02-05T18:00:00  2025-02-12T00:00:00   
4988  BAT-0004989   TMB-DLQ  2025-03-05T18:00:00  2025-03-12T00:00:00   
4989  BAT-0004990   TMB-DLQ  2025-04-05T18:00:00  2025-04-12T00:00:00   
4990  BAT-0004991   TMB-DLQ  2025-05-05T18:00:00  2025-05-12T00:00:00   
4991  BAT-0004992   TMB-DLQ  2025-06-05T18:00:00  2025-06-12T00:00:00   

               survey_method  observation_hours  observed_bat_count  \
0     fixed_acoustic_station               28.6     

In [6]:
df_bats_clean["survey_method"] = df_bats_clean['survey_method'].str.lower().str.strip()

df_bats_clean["survey_end"] = pd.to_datetime(df_bats_clean['survey_end'],errors="coerce")
df_bats_clean['survey_end'] = df_bats_clean['survey_end'].dt.strftime('%Y-%m-%dT%H:%M:%S')

print(df_bats_clean["survey_method"].value_counts())
print(df_bats_clean["survey_end"].value_counts())

# Attention : on a 63 valeurs au lieu de 64 comme tout le reste

survey_method
acoustic_and_visual       1746
fixed_acoustic_station    1625
acoustic_transect         1621
Name: count, dtype: int64
survey_end
2019-01-12T00:00:00    64
2019-03-12T00:00:00    64
2019-05-12T00:00:00    64
2019-06-12T00:00:00    64
2019-07-12T00:00:00    64
                       ..
2021-07-12T00:00:00    63
2021-08-12T00:00:00    63
2022-01-12T00:00:00    63
2024-09-12T00:00:00    63
2025-06-12T00:00:00    63
Name: count, Length: 78, dtype: int64


In [7]:
# On créé 3 catégories à partir de survey_quality
# On garde la colonne survey_quality
dummies = pd.get_dummies(df_bats_clean['survey_quality'], prefix='survey_quality')
df_bats_clean = pd.concat([df_bats_clean, dummies], axis=1)

In [8]:
for c in df_bats_clean:
    print(df_bats_clean[c].value_counts())

survey_id
BAT-0000001    1
BAT-0000002    1
BAT-0000003    1
BAT-0000004    1
BAT-0000005    1
              ..
BAT-0004988    1
BAT-0004989    1
BAT-0004990    1
BAT-0004991    1
BAT-0004992    1
Name: count, Length: 4992, dtype: int64
site_code
KNT-KES    78
KNT-LUM    78
KNT-SEN    78
AVL-MAR    78
AVL-VEL    78
           ..
TMB-WST    78
TMB-SXP    78
TMB-WQL    78
TMB-QCB    78
TMB-DLQ    78
Name: count, Length: 64, dtype: int64
survey_start
2019-01-05T18:00:00    64
2019-02-05T18:00:00    64
2019-03-05T18:00:00    64
2019-04-05T18:00:00    64
2019-05-05T18:00:00    64
                       ..
2025-02-05T18:00:00    64
2025-03-05T18:00:00    64
2025-04-05T18:00:00    64
2025-05-05T18:00:00    64
2025-06-05T18:00:00    64
Name: count, Length: 78, dtype: int64
survey_end
2019-01-12T00:00:00    64
2019-03-12T00:00:00    64
2019-05-12T00:00:00    64
2019-06-12T00:00:00    64
2019-07-12T00:00:00    64
                       ..
2021-07-12T00:00:00    63
2021-08-12T00:00:00    63
2022-